In [0]:
from pyspark.sql.functions import current_timestamp, col, split, element_at, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
CATALOG_NAME = 'beverage_sales'
SCHEMA_NAME = 'bronze'
TABLE_NAME = 'sales'

FILE_PATH = '/Volumes/beverage_sales/bronze/raw_data/abi_bus_case1_beverage_sales_20210726.csv'

In [0]:
SCHEMA = StructType(
    [
        StructField('date', StringType(), True),
        StructField('ce_brand_flvr', StringType(), True),
        StructField('brand_nm', StringType(), True),
        StructField('btlr_org_lvl_c_desc', StringType(), True),
        StructField('chnl_group', StringType(), True),
        StructField('trade_chnl_desc', StringType(), True),
        StructField('pkg_cat', StringType(), True),
        StructField('pkg_cat_desc', StringType(), True),
        StructField('tsr_pckg_nm', StringType(), True),
        StructField('dollar_volume', StringType(), True),
        StructField('year', StringType(), True),
        StructField('month', StringType(), True),
        StructField('period', StringType(), True)
    ]
)

In [0]:
df = spark\
    .read\
    .format('csv')\
    .schema(SCHEMA)\
    .option('header', True)\
    .option('sep', '\t')\
    .option('encoding', 'UTF-16LE')\
    .option('lineSep', '\r\n')\
    .load(FILE_PATH)\
    .withColumn('source_dir', regexp_replace(col('_metadata.file_path'), '/[^/]+$', ''))\
    .withColumn('source_file', element_at(split(col('_metadata.file_path'), '/'), -1))\
    .withColumn('ingestion_timestamp', current_timestamp())

In [0]:
df.write\
    .mode('overwrite')\
    .saveAsTable(f'{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}')